In [ ]:
import timm
import torch.nn as nn
from torchvision.datasets import VOCDetection
import torch
from torchvision.transforms import transforms
from functools import partial

In [ ]:
import sys
from pathlib import Path
import os

In [ ]:
ROOT_DIR = Path.cwd().parent
DATA_DIR = str(ROOT_DIR/'data')

In [ ]:
sys.path.append(str(ROOT_DIR))

In [ ]:
sys.path

In [ ]:
main_dir = Path(ROOT_DIR/'data'/'VOCdevkit'/'VOC2007'/'ImageSets'/'Main')
# Using set to collect single value
CLASSES = sorted({
    f.rsplit("_", 1)[0]
    for f in os.listdir(main_dir)
    if "_" in f and f.endswith(".txt")
})

In [ ]:
CLASSES

In [ ]:
CLASSES2IDX = {cls:(idx+1) for idx, cls in enumerate(CLASSES)}
CLASSES2IDX

In [ ]:
CLASSES2IDX['background'] = 0
CLASSES2IDX

In [ ]:
from src.orgainize_data import *

In [ ]:
model_ = timm.create_model(model_name='resnet18.a1_in1k', pretrained=True)

In [ ]:
X = torch.rand(1,3,224,224)
X.shape

In [ ]:
model_(X)

In [ ]:
for name, module in model_.named_modules():
    # Xét xem module đó có thuộc tính parameter không
    if hasattr(module, 'parameters'):
        is_trainable = any(param.requires_grad for param in module.parameters())
        print(f"Name: {name} - {'trainable' if is_trainable else 'not trainable'}")

In [ ]:
from torchsummary import summary

In [ ]:
summary(model_, input_size=(3,224,224), batch_size=2, device='cpu')

In [ ]:
model_.global_pool = nn.Identity()
model_.fc = nn.Identity()

In [ ]:
summary(model_, input_size=(3,224,224), batch_size=2, device='cpu')

In [ ]:
model_.pretrained_cfg

In [ ]:
from src.orgainize_data import * 

In [ ]:
dataset = VOCObjectDetectionDataset(
    data_dir=DATA_DIR,
    split='trainval',
    label2idx=CLASSES2IDX,
    list_transforms= [transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )]
)

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
collate = partial(collate_fn)

In [ ]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate)

In [ ]:
from model.overfeat_model import *

In [ ]:
ObjectDetectionModel = OverFeatDetector(num_classes=len(CLASSES2IDX))

In [ ]:
trainer = TrainModel(
    batch_size=32,
    dataloader=dataloader,
    model=ObjectDetectionModel
)

In [ ]:
trainer()

In [ ]:
PATH_SAVE_WEIGHT = ROOT_DIR/'model_weight.pth'
torch.save(ObjectDetectionModel.state_dict(), Path(PATH_SAVE_WEIGHT))